# Bank Statement Anomaly Detection — Pilot & Publication Harness (v0.1)

**Purpose:** Compare TabPFN (in-context, no per-client training) against the autoencoder and classical
baselines on transformed bank-statement rows, with a labeled evaluation via typed anomaly injection.

**How to use in the FIS environment:**
1. Set `CFG.data_path` to your transformed data file (parquet/csv/xlsx) and `CFG.use_synthetic = False`.
2. Run all cells top to bottom. Results land in `./results/`.
3. First run works without TabPFN installed (that detector is skipped with a warning).
   To enable it: `pip install tabpfn tabpfn-extensions` (needs torch; one-time model weight download —
   for air-gapped envs, download weights on a connected machine and copy per Prior Labs offline docs).

**Requirements:** pandas, numpy, scikit-learn, pyod (`pip install pyod`), matplotlib. Optional: tabpfn, tabpfn-extensions.

**Publication framing this supports:** per-anomaly-type recall, multi-seed CIs, runtime profiling,
context-size ablation, and a synthetic generator for the reproducibility package.

In [ ]:
# ============ CONFIG ============
from dataclasses import dataclass, field

@dataclass
class Config:
    use_synthetic: bool = True            # False in FIS env
    data_path: str = "transformed_data.parquet"   # parquet/csv/xlsx accepted
    results_dir: str = "results"
    train_frac_by_time: float = 0.8       # temporal split point (quantile of dates)
    injection_rate: float = 0.02          # fraction of test rows corrupted
    seeds: tuple = (0, 1, 2, 3, 4)        # multi-seed for confidence intervals
    flag_budget: float = 0.03             # top-k fraction flagged for per-type recall
    precision_at_k: int = 50
    context_cap: int = 9800               # TabPFN context row cap
    # synthetic generator knobs (ignored when use_synthetic=False)
    syn_accounts: int = 60
    syn_days: int = 730

CFG = Config()
import os; os.makedirs(CFG.results_dir, exist_ok=True)
print(CFG)

## 1. Core library — schema contract, cleaning, synthetic generator, injection, context builder

In [ ]:
"""Core functions for the anomaly detection pilot - tested here, embedded in notebook."""
import numpy as np
import pandas as pd

ID_COLS = ["BankAccountCode", "txn_date", "CurrencyCode"]
FEATURE_COLS = [
    "txn_count", "amount_sum", "amount_mean", "amount_max", "amount_stddev",
    "credit_count", "debit_count", "credit_amount_sum", "debit_amount_sum",
    "unique_BU_count", "weekend_txn_count", "unique_txn_code_count",
    "top_bu_txn_count", "BU_concentration_ratio", "unique_currency_count",
    "credit_debit_ratio", "net_amount",
]

def validate_schema(df):
    issues = []
    missing = [c for c in ID_COLS + FEATURE_COLS if c not in df.columns]
    if missing: issues.append(f"Missing columns: {missing}")
    if not missing:
        if (df["txn_count"] <= 0).any(): issues.append("txn_count has non-positive values")
        bad = (df["credit_count"] + df["debit_count"] != df["txn_count"]).sum()
        if bad: issues.append(f"{bad} rows: credit_count+debit_count != txn_count")
        if df[FEATURE_COLS].isna().any().any():
            issues.append(f"NaNs in: {list(df[FEATURE_COLS].columns[df[FEATURE_COLS].isna().any()])}")
        if np.isinf(df[FEATURE_COLS].select_dtypes(include=[np.number])).any().any():
            issues.append("Infinities present (check credit_debit_ratio when debit_count=0)")
    return issues

def clean_features(df):
    df = df.copy()
    df["txn_date"] = pd.to_datetime(df["txn_date"])
    # recompute concentration as float (sample had int)
    df["BU_concentration_ratio"] = df["top_bu_txn_count"] / df["txn_count"]
    # bounded ratio: credits/(credits+debits) avoids div-by-zero and infinity
    df["credit_debit_ratio"] = df["credit_count"] / (df["credit_count"] + df["debit_count"])
    for c in FEATURE_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype(float)
    df[FEATURE_COLS] = df[FEATURE_COLS].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return df

def make_synthetic(n_accounts=60, n_days=730, seed=0, start="2024-01-01"):
    """Synthetic data mimicking the schema: per-account stable habits + daily noise."""
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start, periods=n_days, freq="D")
    rows = []
    for a in range(n_accounts):
        acct = f"ACC{a:04d}"
        base_txn = rng.integers(2, 40)
        base_amt = 10 ** rng.uniform(3.5, 6.5)
        p_active = rng.uniform(0.15, 0.9)
        credit_share = np.clip(rng.beta(2, 2), 0.05, 0.95)
        n_bu = rng.integers(1, 6); n_code = rng.integers(1, 4)
        for d in dates:
            if rng.random() > p_active: continue
            k = max(1, int(rng.poisson(base_txn)))
            amts = rng.lognormal(np.log(base_amt), 0.6, size=k)
            cc = int(np.round(k * np.clip(credit_share + rng.normal(0, 0.05), 0, 1)))
            dc = k - cc
            camt = amts[:cc].sum(); damt = amts[cc:].sum()
            ubu = min(k, max(1, int(rng.normal(n_bu, 0.5))))
            top_bu = max(1, int(k * rng.uniform(1.0/max(ubu,1), 1.0)))
            wk = k if d.weekday() >= 5 else 0
            rows.append(dict(
                BankAccountCode=acct, txn_date=d, CurrencyCode="USD",
                txn_count=k, amount_sum=amts.sum(), amount_mean=amts.mean(),
                amount_max=amts.max(), amount_stddev=amts.std() if k > 1 else 0.0,
                credit_count=cc, debit_count=dc,
                credit_amount_sum=camt, debit_amount_sum=damt,
                unique_BU_count=ubu, weekend_txn_count=wk,
                unique_txn_code_count=min(k, max(1, int(rng.normal(n_code, 0.4)))),
                top_bu_txn_count=min(top_bu, k), BU_concentration_ratio=0.0,
                unique_currency_count=1, credit_debit_ratio=0.0,
                net_amount=camt - damt,
            ))
    return clean_features(pd.DataFrame(rows))

# ---------------- Anomaly injection ----------------
def _recount(r, new_k):
    old_k = max(r["txn_count"], 1)
    f = new_k / old_k
    r["txn_count"] = new_k
    r["credit_count"] = int(round(r["credit_count"] * f))
    r["debit_count"] = new_k - r["credit_count"]
    r["amount_sum"] *= f; r["credit_amount_sum"] *= f; r["debit_amount_sum"] *= f
    r["top_bu_txn_count"] = min(int(round(r["top_bu_txn_count"] * f)), new_k)
    r["net_amount"] = r["credit_amount_sum"] - r["debit_amount_sum"]
    return r

def inject_volume_spike(r, rng):
    return _recount(r, int(r["txn_count"] * rng.uniform(8, 20)) + 5)

def inject_large_amount(r, rng):
    mult = rng.uniform(15, 60)
    delta = r["amount_max"] * (mult - 1)
    r["amount_max"] *= mult
    r["amount_sum"] += delta
    side = "credit_amount_sum" if rng.random() < 0.5 else "debit_amount_sum"
    r[side] += delta
    r["amount_mean"] = r["amount_sum"] / max(r["txn_count"], 1)
    r["amount_stddev"] = max(r["amount_stddev"], r["amount_max"] * 0.4)
    r["net_amount"] = r["credit_amount_sum"] - r["debit_amount_sum"]
    return r

def inject_one_sided(r, rng):
    k = max(int(r["txn_count"]), 3)
    if rng.random() < 0.5:
        r["credit_count"] = k; r["debit_count"] = 0
        r["credit_amount_sum"] = r["amount_sum"]; r["debit_amount_sum"] = 0.0
    else:
        r["debit_count"] = k; r["credit_count"] = 0
        r["debit_amount_sum"] = r["amount_sum"]; r["credit_amount_sum"] = 0.0
    r["txn_count"] = k
    r["net_amount"] = r["credit_amount_sum"] - r["debit_amount_sum"]
    return r

def inject_bu_concentration(r, rng):
    r["unique_BU_count"] = 1
    r["top_bu_txn_count"] = r["txn_count"]
    return r

def inject_bu_diversity(r, rng):
    k = max(int(r["txn_count"]), 12)
    r["txn_count"] = k
    r["unique_BU_count"] = min(k, int(rng.uniform(10, 20)))
    r["top_bu_txn_count"] = max(1, int(k / r["unique_BU_count"]))
    return r

def inject_code_diversity(r, rng):
    r["unique_txn_code_count"] = min(max(int(r["txn_count"]), 8), int(rng.uniform(8, 15)))
    r["txn_count"] = max(r["txn_count"], r["unique_txn_code_count"])
    return r

def inject_weekend(r, rng):
    r["weekend_txn_count"] = r["txn_count"]
    return r

def inject_net_flow(r, rng):
    boost = r["amount_sum"] * rng.uniform(20, 60)
    if rng.random() < 0.5:
        r["credit_amount_sum"] += boost
    else:
        r["debit_amount_sum"] += boost
    r["amount_sum"] += boost
    r["amount_mean"] = r["amount_sum"] / max(r["txn_count"], 1)
    r["net_amount"] = r["credit_amount_sum"] - r["debit_amount_sum"]
    return r

INJECTORS = {
    "volume_spike": inject_volume_spike,
    "large_amount": inject_large_amount,
    "one_sided_flow": inject_one_sided,
    "bu_concentration": inject_bu_concentration,
    "bu_diversity": inject_bu_diversity,
    "code_diversity": inject_code_diversity,
    "weekend_activity": inject_weekend,
    "net_flow_extreme": inject_net_flow,
}

def inject_anomalies(df, rate=0.02, seed=0):
    rng = np.random.default_rng(seed)
    df = df.copy().reset_index(drop=True)
    df["is_anomaly"] = 0
    df["anomaly_type"] = "none"
    n_inject = max(1, int(len(df) * rate))
    idx = rng.choice(len(df), size=n_inject, replace=False)
    types = list(INJECTORS.keys())
    for i in idx:
        t = types[int(rng.integers(len(types)))]
        row = df.loc[i].copy()
        row = INJECTORS[t](row, rng)
        df.loc[i] = row
        df.loc[i, "is_anomaly"] = 1
        df.loc[i, "anomaly_type"] = t
    # keep derived ratios consistent after injection
    df["BU_concentration_ratio"] = df["top_bu_txn_count"] / df["txn_count"]
    df["credit_debit_ratio"] = df["credit_count"] / (df["credit_count"] + df["debit_count"])
    df["credit_debit_ratio"] = df["credit_debit_ratio"].fillna(0.0)
    return df

def build_context(day_df, history_df, cap=9800):
    """Production-style context: history of only the accounts present today, newest first."""
    accts = day_df["BankAccountCode"].unique()
    ctx = history_df[history_df["BankAccountCode"].isin(accts)]
    ctx = ctx.sort_values("txn_date", ascending=False).head(cap)
    return ctx


## 2. Load data

Validation runs first; fix any reported issue before trusting results.
Note two intentional recomputations in `clean_features`:
- `BU_concentration_ratio` recomputed as float (`top_bu_txn_count / txn_count`) — the sample file had it as int.
- `credit_debit_ratio` redefined as `credit_count / (credit_count + debit_count)` — bounded 0..1, no infinity when `debit_count = 0`.

In [ ]:
import pandas as pd, numpy as np

if CFG.use_synthetic:
    df = make_synthetic(CFG.syn_accounts, CFG.syn_days, seed=42)
    print("SYNTHETIC data:", len(df), "rows")
else:
    p = CFG.data_path
    raw = pd.read_parquet(p) if p.endswith("parquet") else (pd.read_excel(p) if p.endswith("xlsx") else pd.read_csv(p))
    issues = validate_schema(raw)
    print("Validation issues:", issues if issues else "none")
    df = clean_features(raw)
    print("REAL data:", len(df), "rows,", df.BankAccountCode.nunique(), "accounts,",
          df.txn_date.min().date(), "to", df.txn_date.max().date())

display(df.head(3))

## 3. Quick data profile (goes into the paper's data section)

In [ ]:
print("rows per account: ", df.groupby("BankAccountCode").size().describe().round(1).to_dict())
print("accounts active per day:", df.groupby("txn_date").size().describe().round(1).to_dict())
print("txn_count distribution:", df.txn_count.describe().round(1).to_dict())
const_cols = [c for c in FEATURE_COLS if df[c].std() < 1e-9]
print("constant (zero-variance) features:", const_cols, " <- these carry no signal and are dropped later")

## 4. Temporal split + anomaly injection

Split is by date (never random - avoids leakage). Injection corrupts a known fraction of **test** rows
with one of 8 typed corruptions, each mapping to an anomaly type from the design document.
Injections modify correlated features jointly, so they are not trivially detectable single-feature edits.

In [ ]:
split_date = df["txn_date"].quantile(CFG.train_frac_by_time)
train = df[df["txn_date"] <= split_date].copy()
test_clean = df[df["txn_date"] > split_date].copy()
print(f"train: {len(train)} rows to {split_date.date()} | test: {len(test_clean)} rows after")

test = inject_anomalies(test_clean, rate=CFG.injection_rate, seed=0)
print("injected:", test.is_anomaly.sum())
print(test[test.is_anomaly==1].anomaly_type.value_counts().to_string())

## 5. Feature prep

log1p on heavy-tailed amount features (your sample spans 5 orders of magnitude), signed-log on net_amount,
drop zero-variance columns, standardize using train statistics only.

In [ ]:
from sklearn.preprocessing import StandardScaler

AMOUNT = ["amount_sum","amount_mean","amount_max","amount_stddev","credit_amount_sum","debit_amount_sum"]
def prep(d):
    X = d[FEATURE_COLS].copy()
    for c in AMOUNT: X[c] = np.log1p(X[c].clip(lower=0))
    X["net_amount"] = np.sign(X["net_amount"]) * np.log1p(X["net_amount"].abs())
    return X

Ptr = prep(train)
KEEP = list(Ptr.columns[Ptr.std() > 1e-9])
print("using", len(KEEP), "features; dropped:", [c for c in Ptr.columns if c not in KEEP])
scaler = StandardScaler().fit(Ptr[KEEP])
def transform(d): return scaler.transform(prep(d)[KEEP])

## 6. Detectors

All expose `fit(Xtr)` + `score(Xte)` where **higher score = more anomalous**.
The MLP autoencoder is a portable stand-in — swap in your production torch AE via the same wrapper.
TabPFN runs in two modes: `global` (train rows as context) and `per_account`
(each test row scored against its own account's history — the production pattern and the paper's key comparison).

In [ ]:
import time, warnings
warnings.filterwarnings("ignore")
from sklearn.neural_network import MLPRegressor
from pyod.models.iforest import IForest
from pyod.models.ecod import ECOD
from pyod.models.copod import COPOD
from pyod.models.lof import LOF
from pyod.models.pca import PCA as PCAOD

class AEWrap:
    name = "Autoencoder(MLP)"
    def __init__(self, seed=0):
        self.m = MLPRegressor(hidden_layer_sizes=(12,5,12), max_iter=400, random_state=seed)
    def fit(self, X): self.m.fit(X, X); return self
    def score(self, X): return ((self.m.predict(X) - X)**2).mean(axis=1)

class PyODWrap:
    def __init__(self, name, model): self.name, self.m = name, model
    def fit(self, X): self.m.fit(X); return self
    def score(self, X): return np.nan_to_num(self.m.decision_function(X), nan=0.0, posinf=1e12, neginf=-1e12)

def make_detectors(seed=0):
    return [AEWrap(seed),
            PyODWrap("IForest", IForest(random_state=seed)),
            PyODWrap("ECOD", ECOD()),
            PyODWrap("COPOD", COPOD()),
            PyODWrap("LOF", LOF()),
            PyODWrap("PCA", PCAOD())]

# ---- TabPFN (optional; verify API version in FIS env) ----
TABPFN_OK = False
try:
    from tabpfn_extensions import unsupervised as tab_unsup
    from tabpfn import TabPFNClassifier, TabPFNRegressor
    TABPFN_OK = True
    print("TabPFN available")
except Exception as e:
    print("TabPFN not available - skipping that detector. Install: pip install tabpfn tabpfn-extensions")

def tabpfn_scores(ctx_X, test_X):
    """Negative log-likelihood of test rows under context distribution (higher = more anomalous).
    NOTE: API surface of tabpfn-extensions changes between versions. If .outliers() is absent,
    check dir(model) for the density/outlier scoring method in your installed version."""
    model = tab_unsup.TabPFNUnsupervisedModel(TabPFNClassifier(), TabPFNRegressor())
    import torch
    model.fit(torch.as_tensor(np.asarray(ctx_X), dtype=torch.float32))
    s = model.outliers(torch.as_tensor(np.asarray(test_X), dtype=torch.float32))
    s = np.asarray(s, dtype=float).ravel()
    return np.nan_to_num(-s if np.nanmedian(s) > 0 else s, nan=0.0)  # ensure higher=anomalous; verify sign on known injections!

## 7. Evaluation — multi-seed benchmark

Each seed re-injects anomalies and re-fits every detector. Outputs mean ± std for ROC-AUC, PR-AUC,
precision@k, plus fit/score wall-clock (the deployment-cost evidence).

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

def evaluate(scores, y, k):
    order = np.argsort(-scores)
    return dict(roc_auc=roc_auc_score(y, scores),
                pr_auc=average_precision_score(y, scores),
                p_at_k=float(y[order[:k]].mean()))

records = []
for seed in CFG.seeds:
    te = inject_anomalies(test_clean, rate=CFG.injection_rate, seed=seed)
    Xtr, Xte, y = transform(train), transform(te), te.is_anomaly.values
    for det in make_detectors(seed):
        t0 = time.perf_counter(); det.fit(Xtr); t_fit = time.perf_counter() - t0
        t0 = time.perf_counter(); s = det.score(Xte); t_score = time.perf_counter() - t0
        records.append(dict(seed=seed, model=det.name, fit_s=t_fit, score_s=t_score,
                            **evaluate(s, y, CFG.precision_at_k)))
    if TABPFN_OK:
        t0 = time.perf_counter()
        s = tabpfn_scores(Xtr[:CFG.context_cap], Xte)
        records.append(dict(seed=seed, model="TabPFN(global)", fit_s=0.0,
                            score_s=time.perf_counter()-t0, **evaluate(s, y, CFG.precision_at_k)))

res = pd.DataFrame(records)
summary = res.groupby("model").agg(["mean","std"]).round(3)
summary.to_csv(f"{CFG.results_dir}/benchmark_summary.csv")
display(summary[["roc_auc","pr_auc","p_at_k","fit_s","score_s"]])

## 8. Per-anomaly-type recall

The paper's most interesting table: which detector catches which anomaly class at a fixed 3% flag budget.
Expect account-relative types (bu_concentration, weekend_activity) to be hard for global detectors —
that gap is the motivation for per-account context.

In [ ]:
te = inject_anomalies(test_clean, rate=CFG.injection_rate, seed=0)
Xtr, Xte = transform(train), transform(te)
budget = int(CFG.flag_budget * len(te))
rows = {}
for det in make_detectors(0):
    s = det.fit(Xtr).score(Xte)
    flag = np.zeros(len(te)); flag[np.argsort(-s)[:budget]] = 1
    t = te.reset_index(drop=True).assign(flag=flag)
    rows[det.name] = t[t.is_anomaly==1].groupby("anomaly_type").flag.mean()
per_type = pd.DataFrame(rows).round(2)
per_type.to_csv(f"{CFG.results_dir}/per_type_recall.csv")
display(per_type)

ax = per_type.plot(kind="bar", figsize=(11,4), width=0.8)
ax.set_ylabel(f"recall @ top {int(CFG.flag_budget*100)}%"); ax.set_title("Per-anomaly-type recall by detector")
import matplotlib.pyplot as plt; plt.xticks(rotation=30, ha="right"); plt.tight_layout()
plt.savefig(f"{CFG.results_dir}/per_type_recall.png", dpi=140); plt.show()

## 9. TabPFN per-account context mode (production pattern)

Scores each test day's rows against the history of **only the accounts present that day** (capped),
exactly like the planned daily inference. Slow (one TabPFN call per day) — run on a subset first.

In [ ]:
if TABPFN_OK:
    te = inject_anomalies(test_clean, rate=CFG.injection_rate, seed=0)
    days = sorted(te.txn_date.unique())
    all_scores, all_idx = [], []
    for d in days:
        day_rows = te[te.txn_date == d]
        ctx = build_context(day_rows, train, cap=CFG.context_cap)
        if len(ctx) < 50: continue                     # too little context to judge
        s = tabpfn_scores(transform(ctx), transform(day_rows))
        all_scores.append(s); all_idx.extend(day_rows.index)
    s = np.concatenate(all_scores); y = te.loc[all_idx, "is_anomaly"].values
    print("TabPFN(per_account):", evaluate(s, y, CFG.precision_at_k))
else:
    print("TabPFN not installed - skipped")

## 10. Context-size ablation (data-efficiency evidence for the paper)

In [ ]:
ablation = []
te = inject_anomalies(test_clean, rate=CFG.injection_rate, seed=0)
Xte, y = transform(te), te.is_anomaly.values
for n in [500, 1000, 3000, 6000, len(train)]:
    sub = train.sort_values("txn_date").tail(n)
    Xs = transform(sub)
    for det in [AEWrap(0), PyODWrap("LOF", LOF())]:
        s = det.fit(Xs).score(Xte)
        ablation.append(dict(context_rows=n, model=det.name, **evaluate(s, y, CFG.precision_at_k)))
    if TABPFN_OK:
        s = tabpfn_scores(Xs[:CFG.context_cap], Xte)
        ablation.append(dict(context_rows=n, model="TabPFN(global)", **evaluate(s, y, CFG.precision_at_k)))
ab = pd.DataFrame(ablation)
ab.to_csv(f"{CFG.results_dir}/context_ablation.csv", index=False)
display(ab.pivot(index="context_rows", columns="model", values="pr_auc").round(3))

## 11. Analyst review export

Top-k flagged **genuine** (non-injected) rows for analyst validation — the real-data precision@k layer.

In [ ]:
det = AEWrap(0).fit(transform(train))
genuine = test_clean.copy()
genuine["score"] = det.score(transform(genuine))
top = genuine.nlargest(20, "score")[ID_COLS + ["score"] + FEATURE_COLS]
top.to_csv(f"{CFG.results_dir}/analyst_review_top20.csv", index=False)
print("exported results/analyst_review_top20.csv - have analysts mark each plausible / not plausible")
display(top.head(5))

## Next steps
1. **FIS run 1:** real data, `use_synthetic=False`, cells 1-8 (no TabPFN needed yet). Send me the two CSVs + profile numbers.
2. **FIS run 2:** install tabpfn + tabpfn-extensions, verify the `tabpfn_scores` API against your installed version (see docstring), run cells 9-10.
3. **Analyst layer:** circulate `analyst_review_top20.csv`.
4. Then we harden injections against the real data's distribution, add drift evaluation (score later months without refresh), and start the paper draft.